# torch.distributed.new_group()
- torch.distributed.new_group 是 PyTorch 分布式通信模块（torch.distributed）中的一个核心接口，<font color='red'>主要用于创建自定义的进程组（Process Group）。</font>

- <font color='red'>在分布式训练中，默认情况下所有进程（即整个 world）都在同一个全局通信组内。</font><font color='green'>但在一些复杂的并行策略（如大模型训练中的张量并行、流水线并行）或特定的通信需求下，我们需要让一部分进程进行细粒度的通信，这时就需要用到 new_group</font>

## 🛠️ 核心功能与参数
它的基本用法是通过指定一部分进程的 rank（进程编号）来<font color='red'>创建一个子组。</font>

In [ ]:
import torch.distributed as dist

# 创建一个包含 rank 0 和 rank 1 的通信组
ranks = [0, 1]
group = dist.new_group(ranks, backend='nccl')

### 主要参数说明：
- ranks：一个整数列表，<font color='red'>指定哪些进程（rank）属于这个新的通信组。</font>
- backend：<font color='red'>指定该组使用的通信后端，如 'nccl'（推荐用于 GPU 通信）</font>或 'gloo'（常用于 CPU 通信）。<font color='red'>如果不指定，默认使用全局进程组的后端。</font>
- timeout：组内操作（如集合通信）的超时时间，默认通常是 30 分钟。

## 💡 核心作用
### 1.细粒度的通信控制：
- 在使用集合通信算子（如 all_reduce, all_gather, barrier 等）时，可以通过 group 参数指定通信范围。
- <font color='red'>这样，数据交换和同步就只会发生在该组内的 rank 之间，而不是等待所有进程。</font>

In [ ]:
# 只有指定 group 内的 rank 会在这里进行同步
dist.all_reduce(tensor, group=group)
dist.barrier(group=group)

### 2.点对点通信（P2P）的标识符：
- 如果两个 rank 之间需要同时进行多组不同的点对点通信（send / recv），不同的 group 可以作为唯一的通信标识符，用来区分不同的通信流，避免数据错乱。

### 3.混合后端通信：
- <font color='red'>你可以创建多个通信组，并指定不同的后端。</font>例如，:
    - 某些组使用 nccl 进行高效的 GPU 张量通信，
    - 而另一些组使用 gloo 进行 CPU 张量通信。
    
## ⚠️ 关键规则与常见误区（非常重要）
在使用 new_group 时，新手最容易踩的坑就是死锁（Deadlock）或 NCCL 错误。PyTorch 对创建通信组有两个强制要求：

- 全局参与：所有参与了全局初始化的进程（即整个 world 中的所有 rank），都必须执行 new_group 这行代码，即使某些进程并不属于这个新创建的组。
- 顺序一致：所有进程创建通信组的顺序必须完全相同。

## ❌ 错误的写法（会导致死锁或报错）：
- 原因： 创建子组时，底层（如 NCCL）需要生成一个唯一的通信 ID，并在所有进程间进行同步。如果 rank 2 和 3 跳过了创建步骤，通信 ID 就无法达成一致，导致程序卡死或崩溃。

In [ ]:
# 错误示例：只有组内成员才执行 new_group
if rank in [0, 1]:
    subgroup = dist.new_group([0, 1])

## ✅ 正确的写法：

In [ ]:
group = None
rank = dist.get_rank()

# 所有进程都必须按相同的顺序执行这段代码
ranks_in_group = [0, 1]
subgroup = dist.new_group(ranks_in_group)

# 只有组内成员才保留该组的句柄，用于后续通信
if rank in ranks_in_group:
    group = subgroup

### 如果你需要创建多个通信组，也必须保证所有进程按照完全相同的顺序依次创建：
- 通过这种机制，PyTorch 能够确保分布式环境下的通信拓扑被正确、安全地建立起来。

In [1]:
group1 = dist.new_group([0, 1])
group2 = dist.new_group([2, 3])
group3 = dist.new_group([1, 2, 3])
# 所有进程都会按顺序创建 group1, group2, group3
# 然后各自判断自己属于哪个组，并保存对应的句柄

NameError: name 'dist' is not defined